In [3]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)

from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../oauth.json'
CLIENT_FILE = '../client_auth.json'

PLAYLIST_TSV_DIR = '../playlists/'

In [4]:
Y = YTMusicPlaylists(header=HEADER_FILE, client=CLIENT_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using header file: ../oauth.json
Parsing client auth file: ../client_auth.json
Test Passed in 5.99 seconds
Using ytmusicapi version: 1.10.0
Loaded 499 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [ ]:
playlist_names = [
  'zp john prine radio'
]
MIN_RADIO_LIKE_TO_SPLIT=2
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_radio_playlist(pl_info, 
        move_like=True, create_like_playlist=True, remove_dislike=True, remove_not_like=True,
        min_num_like=MIN_RADIO_LIKE_TO_SPLIT,  sleep=1, verbose=True,
    )


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [ ]:
playlist_names = [
    'brass radio'
]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=USE_CACHE)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


Loaded 333395 playounts from 127381 tracks
Created sorted pl: brass radio [11-29-2024] PLWptjpDqazOxUkSyQ6DYwqndUyllmNfhk, and  deleted original pl: PLWptjpDqazOwwLfVX1XVEbyCxI_5r9wEB


## Save playlist backup tsv

In [13]:
PLAYLIST_NAME = 'zz not like 3'
USE_CACHE = False
_pl_id = Y.query_by_title(PLAYLIST_NAME).playlistId
_pl_tracks, _pl_metadata = Y.save_playlist_tsv(Y.playlist_get_info(_pl_id, use_cache=USE_CACHE))

                                                   0
owned                                           True
id                PLWptjpDqazOxXQs5amrReasOgAL-UC1HF
privacy                                      PRIVATE
description                                     None
views                                           None
duration                                    6+ hours
trackCount                                       123
title                                  zz not like 3
artists                                           []
year                                            2025
related                                           []
duration_seconds                               27951


## Update _not_like tsv

In [14]:
not_like_tracks = Y.collect_all_not_like_tracks_from_tsvs()
not_like_tracks.to_csv(Y.not_like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to not_like tsv: {Y.not_like_tsv}')

Found 3 not like playlists out of the 544 total
Updated not liked tracks, contains 8285 entries.
Saved 8285 entries to not_like tsv: ../playlists/_not_liked_tracks.tsv


## Update _like tsv and print like coverage

In [15]:
like_tracks = Y.collect_all_like_tracks_from_tsvs()
like_tracks.to_csv(Y.like_tsv, sep='\t', header=True)
print(f'Saved {len(like_tracks)} entries to like tsv: {Y.like_tsv}')

Found 274 like playlists out of the 544 total
Beats Lofi.tsv	95.7% currently liked (of 141 total tracks)
Bossa Nova.tsv	96.0% currently liked (of 25 total tracks)
Brass n chill.tsv	96.5% currently liked (of 258 total tracks)
Chillwave.tsv	97.0% currently liked (of 367 total tracks)
Folk.tsv	98.7% currently liked (of 387 total tracks)
Grunge.tsv	98.1% currently liked (of 103 total tracks)
Hip Hop 1980s.tsv	100.0% currently liked (of 25 total tracks)
Hip Hop 1990s G funk.tsv	100.0% currently liked (of 28 total tracks)
Hip Hop 1990s east coast.tsv	100.0% currently liked (of 29 total tracks)
Hip Hop 1990s west Coast.tsv	100.0% currently liked (of 73 total tracks)
Hip Hop 1990s.tsv	98.0% currently liked (of 704 total tracks)
Hip Hop 2000s southern.tsv	96.0% currently liked (of 25 total tracks)
Hip Hop 2000s.tsv	100.0% currently liked (of 92 total tracks)
Hip Hop Hits.tsv	99.6% currently liked (of 240 total tracks)
Indie 1990s Rock.tsv	97.4% currently liked (of 76 total tracks)
Indie 2000s.t

## Get Public Playlists

In [34]:
skip_if_contains=(' radio', 'ym ', ' albums', 'zz not', 'zzz ', 'yyz ', 'Episodes for ', 'Liked Music')
priv = 'PUBLIC'
pub_playlists = Y.get_playlists_by_privacy(Y, privacy=priv, skip_if_contains=skip_if_contains)
print(f'\nFound {len(pub_playlists)} {priv} playlists')

# last run 1-2025, 270 public playlists, fine descriptions

Found public playlist named: Liked Music, description: Auto playlist
Found public playlist named: Folk, description: Jake G • 379 tracks
Found public playlist named: rock krautrock, description: Jake G • 213 tracks
Found public playlist named: electronic witch house, description: Jake G • 64 tracks
Found public playlist named: oldies fallout 40s, description: Jake G • 67 tracks
Found public playlist named: jazz, description: Jake G • 1,299 tracks
Found public playlist named: xr idm, description: Jake G • 112 tracks
Found public playlist named: brass beats, description: Jake G • 210 tracks
Found public playlist named: xr VintageObscura, description: Jake G • 152 tracks
Found public playlist named: ambient grouper, description: Jake G • 72 tracks
Found public playlist named: ambient japan, description: Jake G • 50 tracks
Found public playlist named: rockabilly, description: Jake G • 81 tracks
Found public playlist named: xr indierock, description: Jake G • 61 tracks
Found public playlist

## Get Playlist Counts

In [16]:
%%time
playlist_file = os.path.join(PLAYLIST_TSV_DIR, '_playlist_radio_counts.tsv')
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
playlists.to_csv(playlist_file, sep='\t', index=False)
# Note: now part of ytmusic backup

playlists.head(20)
# last run 8-2024

Getting playlist track count for playlists (takes ~5 minutes)
Only for playlists with radio in the title
CPU times: total: 29.5 s
Wall time: 4min 49s


,title,track_count,privacy,playlist_id
104,xr GypsyJazz radio,18,PRIVATE,PLWptjpDqazOx1ODmPKJHgKuvUD8Grn9i0
87,xr ClassicRock radio,21,PRIVATE,PLWptjpDqazOzSINwC8Q_p6Q7xXYgNGVaS
141,xr SoundsVintage radio,23,PRIVATE,PLWptjpDqazOyjs9o3hDBfVn185qGL8tQ1
132,xr PsychedelicRock radio,36,PRIVATE,PLWptjpDqazOzCK0de7_tgdlcATRlua1wu
140,xr Samplehunters radio,37,PRIVATE,PLWptjpDqazOxZoMxg-ecOwAMbM8Xkpb3x
124,xr MusicToSleepTo radio,39,PRIVATE,PLWptjpDqazOzfumkZJ8sZi_fKVxXLZOfI
0,ambient modern radio,45,PRIVATE,PLWptjpDqazOwO_d5ayc93-4pTuls5NCxG
142,xr SpaceMusic radio,46,PRIVATE,PLWptjpDqazOxGj_rV9vIPsASmecw_1G9Q
37,jazz for spies radio,53,PRIVATE,PLWptjpDqazOyj2ARv0edtYtxjiscELzT3
35,Instrumentals Acoustic radio,55,PRIVATE,PLWptjpDqazOwMXA073PbBvjxkW0VTEC05


# One Off

## Upload playlist from tsv backup

In [7]:
PLAYLIST_NAME = 'zz not like'
for y in [2020, 2021, 2022, 2023, 2024]:
  PLAYLIST_NAME = f'y {y} top'
  playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
  Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)


Generating y 2020 top ytmusic playlist for 92 tracks
Updated y 2020 top playlist with 92 tracks, playlist id: PLWptjpDqazOzDlEPo4KiY-84dER47evqk...waiting 3 seconds...

Generating y 2021 top ytmusic playlist for 66 tracks
Updated y 2021 top playlist with 66 tracks, playlist id: PLWptjpDqazOwSHciGis47Qr0LwQ72i96K...waiting 3 seconds...

Generating y 2022 top ytmusic playlist for 25 tracks
Updated y 2022 top playlist with 25 tracks, playlist id: PLWptjpDqazOxu6dMj1wAKjULe0YWi2a7-...waiting 3 seconds...

Generating y 2023 top ytmusic playlist for 281 tracks
Updated y 2023 top playlist with 281 tracks, playlist id: PLWptjpDqazOycWD3T5uF7iv7ZVkd8uvW0...waiting 3 seconds...

Generating y 2024 top ytmusic playlist for 21 tracks
Updated y 2024 top playlist with 21 tracks, playlist id: PLWptjpDqazOweIE55u2Io89UN-zVg1bYg...waiting 3 seconds...


### Playlist radio and like mapping helpers

In [ ]:
# Get possible tracks to add to radio like playlist map
import pandas as pd
radio_map = pd.read_csv('..\\playlists\\_ytmusic_radio_to_like_pl_map.tsv', sep='\t')
radio_map = dict(zip(radio_map['radio_playlist'], radio_map['like_playlist']))
# radio_map
all_playlists = frozenset(Y.playlists['title'].unique())
add_to_radio_map = []
for t in sorted(all_playlists):
  if not t.endswith("radio"):
    continue
  if t in radio_map:
    continue  
  if t.replace('radio', 'like') in all_playlists:
   continue
  add_to_radio_map.append(t)
  print(t)


Rock modern The New Age of Classic radio
christmas crooners radio
hiphop get dumb radio
instrumental ambient piano radio
instrumental piano classical radio
punk skate or die radio
xr AtmosphericDnB radio
xr DoomMetal radio
xr Flamenco radio
xr Instrumentals radio
xr ModernRockMusic radio
xr OutlawCountry radio
xr SpaceMusic radio
xr Techno radio
xr TropicalHouse radio
xr acidhouse radio
xr animemusic radio
xr bossanova radio
xr chicagohouse radio
xr melodichouse radio
xs DEZKO ASCEND radio
xs Discord Favorites radio
xs Liquid Drum And Bass radio
zp Dilla radio
['Rock modern The New Age of Classic radio', 'christmas crooners radio', 'hiphop get dumb radio', 'instrumental ambient piano radio', 'instrumental piano classical radio', 'punk skate or die radio', 'xr AtmosphericDnB radio', 'xr DoomMetal radio', 'xr Flamenco radio', 'xr Instrumentals radio', 'xr ModernRockMusic radio', 'xr OutlawCountry radio', 'xr SpaceMusic radio', 'xr Techno radio', 'xr TropicalHouse radio', 'xr acidhouse ra

In [ ]:
# Get possible like playlists, then manually edit and add to _like_playlist.tsv
for _, row in Y.playlists.iterrows():
  t = row['title']
  if not (t.endswith("like") or t.endswith("radio") or t.endswith("albums")):
    print(t)

Liked Music
2022 Recap
2023 Recap
ambiant electro
ambient
ambient BOC
ambient Dream Pop Deep Sleep
ambient haunting harmonious
ambient Indie Synths
ambient japan
beats
beats 2010s wonky LA scene
beats cosmic Slop
beats dj
beats instrumental
Beats Lofi
beats Soulful Instrumentals
bluegrass billy
blues
blues chicago
blues delta roots
blues electric
blues texas roots
Bossa Nova
Brass n chill
Chill Supermix
Chillwave
Discover Weekly
electronic
electronic 1990s electronica
electronic 1990s late 1980s House
electronic 2000s
electronic 2010s
electronic Analog Grooves
electronic big beats
electronic chill
electronic Dance
electronic deep house
electronic dubstep uk
electronic Focus
electronic hard dj
electronic house
electronic house big bass
electronic house dj
electronic house french touch
electronic house funk
electronic House Soulful
electronic House Special
electronic indie essentials
electronic Innerwaves
electronic jaar
electronic new indie beats
electronic soft pad
electronic trance dj

## Query

In [ ]:
Y.query_by_title(playlist_name)

title                                                 jazz radio
playlistId                    PLWptjpDqazOxgrXKM4xfJ-SBRahEDfXmG
thumbnails     [{'url': 'https://yt3.googleusercontent.com/zS...
description                                  Jake G • 797 tracks
count                                                        797
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
Name: 20, dtype: object

### Rename Batch Playlists

In [ ]:
VERBOSE = True
DRY_RUN = True

# Rename original subreddit playslists tor shorter no . or _ name
# See: notebooks/old/ytmusic_rename_some_playlists.ipynb
# Example: x_r.bluegrass_tracks_like --> xr bluegrass like
for i, row in Y.playlists.iterrows():
    playlist_name = row['title']
    if playlist_name.startswith("x_r."):
        new_name = playlist_name.replace("x_r.", "xr ").replace("_tracks_", " ").replace("_albums", " albums") 
        if VERBOSE:
            print(f"Renaming: {playlist_name} --> {new_name}")
        if not DRY_RUN:
            Y.rename_playlist(row['playlistId'], new_name)
            
# Manually fix these
Y.find_playlists_with_special_chars(special_chars=["_", "/", ".", ":", ",", "x.r", "x.s" "y_", "x.sp", "Produced", "p_"])

Renaming: x_r.2000smusic_tracks_like --> xr 2000smusic like
Renamed playlist with ID 'PLWptjpDqazOxnmxikFDKOXszYFONGWs8U' to 'xr 2000smusic like'
Renaming: x_r.2000smusic_tracks_radio --> xr 2000smusic radio
Renamed playlist with ID 'PLWptjpDqazOzwVaW_CNc6xtp7v_stZ_dX' to 'xr 2000smusic radio'
Renaming: x_r.2010smusic_tracks_like --> xr 2010smusic like
Renamed playlist with ID 'PLWptjpDqazOwWltLPE8IThgPJeElLQIUO' to 'xr 2010smusic like'
Renaming: x_r.50sMusic_tracks_like --> xr 50sMusic like
Renamed playlist with ID 'PLWptjpDqazOwCLs4ZOeyb_Lg4IgAGlg2L' to 'xr 50sMusic like'
Renaming: x_r.60sMusic_tracks_like --> xr 60sMusic like
Renamed playlist with ID 'PLWptjpDqazOzTAvihnt0PZpLX-bj0JJh2' to 'xr 60sMusic like'
Renaming: x_r.70sMusic_albums --> xr 70sMusic albums
Renamed playlist with ID 'PLWptjpDqazOyo7RbE6Pjv1HJUa61TRhPc' to 'xr 70sMusic albums'
Renaming: x_r.70sMusic_tracks_like --> xr 70sMusic like
Renamed playlist with ID 'PLWptjpDqazOyxy1ElEB7K1qG2A9Pie5ar' to 'xr 70sMusic like'


### Generate playlist froma a list of albums

In [4]:
# ORIG
# name = 'y_2023_albums_to_listen_to_v3'
# desc = 'manually selected albums to top off 2023 albums'
# albums_to_add = [
# "Aesop Rock - Integrated Tech Solutions",
# "Mitski - The Land Is Inhospitable and So Are We",
# "Olivia Rodrigo - GUTS",
# "The National - First Two Pages Of Frankenstein",
# "Foo Fighters - But Here We Are",
# "Jessie Ware - That! Feels Good!",
# ]


# TODO run this before playlistbackup
# Playlist for all mb library albums
dry_run = False
verbose = False
uni_module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if uni_module_path not in sys.path: sys.path.append(uni_module_path)
import unify_lib as uni
import pandas as pd
import time

YT_ALBUM_MATCH_TSV = os.path.join(uni_module_path, 'tsvs', 'ytmusic_musicbee_album_matches.tsv')

DATE = time.strftime('%m-%d-%Y')
MAX_PL_TRACKS = 3000

# Just do library
MB_LIB = os.path.join(uni_module_path, 'db_assets', 'musicbee_library.tsv')

# MB_INBOX = os.path.join(uni_module_path, 'db_assets', 'musicbee_inbox.tsv')
name = 'za mb library unmatched albums'
desc = f' albums in mb library but not not in ytmusic_musicbee_album_matches.tsv {DATE}'


print('Loading ytmusic and musicbee library track databases (takes ~1m)')
# mb_tracks = uni.ingest_musicbee_db_assets(MB_LIB, MB_INBOX)
mb_tracks = pd.read_csv(MB_LIB, sep='\t')
mb_tracks['fuzzy_album_id'] = mb_tracks.apply(uni.make_mb_fuzzy_album_id, axis=1)
albums_to_add = frozenset(mb_tracks['fuzzy_album_id'].unique())
initial_num_albums = len(albums_to_add)  # Calculate BEFORE removing any albums

last_album_matches = pd.read_csv(YT_ALBUM_MATCH_TSV, sep='\t', index_col=0, low_memory=False)
last_album_matches = last_album_matches.loc[last_album_matches['mb_match_label'] == 'MATCH']
last_album_matches = frozenset(last_album_matches['mb_match'].unique())

albums_removed = albums_to_add.intersection(last_album_matches)
num_removed = len(albums_removed)
albums_to_add -= last_album_matches 
print(f'Loaded {initial_num_albums} unique musicbee albums')
print(f'Loaded {len(last_album_matches)} unique previous matched mb albums already in yt library')
print(f"Removed {num_removed} albums ({(num_removed / initial_num_albums):.0%}) already in last_album_matches.")
print(f'Attempting to add {len(albums_to_add)} albums to ytmusic playlist')

track_ids = []
skipped = set()
for i, a in enumerate(albums_to_add):
    res = Y.yt.search(query=a, filter='albums', limit=1)
    if len(res) == 0 or res[0].get('browseId') == None:
        a2 = a.split(' [')[0].split(' (')[0]
        res = Y.yt.search(query=a2, filter='albums', limit=1)
        if len(res) == 0 or res[0].get('browseId') == None:
          print(f'Skipping query: {a} empty album match result: {res}')
          skipped.add(a)
          continue
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    if verbose:
      print(f'({i+1}) Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album.lower()}')

print(f'Accumulated {len(track_ids)} track ids')

if not dry_run:
  for i in range((len(track_ids) + MAX_PL_TRACKS - 1) // MAX_PL_TRACKS):
      chunk_name = f"{name} {i+1}" if len(track_ids) > MAX_PL_TRACKS else name
      vids = track_ids[i*MAX_PL_TRACKS:(i+1)*MAX_PL_TRACKS]
      pl_id = Y.playlist_from_yt_vids(vids, pl_name=chunk_name, sleep=0.5, public='PRIVATE', desc=desc,
                              dry=dry_run, set_rating=None, remove_dupes=False, verbose=False)
      
      print(f'Generated playlist: {chunk_name} with id {pl_id}')

Loading ytmusic and musicbee library track databases (takes ~1m)
Loaded 5215 unique musicbee albums
Loaded 4630 unique previous matched mb albums already in yt library
Removed 4002 albums (77%) already in last_album_matches.
Attempting to add 1213 albums to ytmusic playlist
Skipping query: shlohmo - shlomoshun redux bad result: []
Skipping query: afx - analord 09 bad result: []
Skipping query: various artists - sunset of a clown volume 2 bad result: []
Skipping query: eats everything & totally enormous extinct dinosaurs - get lost vi - mixed by totally enormous extinct dinosaurs bad result: []
Skipping query: avey tare, panda bear & geologist - danse manatee bad result: []
Skipping query: mary anne hobbs - warrior dubz bad result: []
Skipping query: elvis costello & the attractions - armed forces {1986 imp} bad result: []
Skipping query: christophe - get lost vi - mixed by totally enormous extinct dinosaurs bad result: []
Skipping query: heldon - dj-kicks: four tet bad result: []
Skipp

### Publicize Like Playlists

In [9]:
# Get possible tracks to add to radio like playlist map
import pandas as pd
from pprint import pprint
import time

DATE = time.strftime('%m-%d-%Y')

like_playlists = frozenset(pd.read_csv('..\\playlists\\_like_playlists.tsv', sep='\t')['title'])
radio_map = pd.read_csv('..\\playlists\\_ytmusic_radio_to_like_pl_map.tsv', sep='\t')
like_playlists_mapped = frozenset(radio_map['like_playlist'])
# radio_map = dict(zip(radio_map['radio_playlist'], radio_map['like_playlist']))


#### Run regularly to update privacy to public

maxes if if too many set to public in one day

In [10]:
dry_run = False
print_warn = False
set_public = True # maxes out quickly
min_count_for_public = 15
finished = []
for i, pl in Y.playlists.iterrows():
  p = pl['title']
  if p not in like_playlists:
    if 'like' in p and  not 'not like' in p:
      print('Skipping not in like_playlist:', p)
    continue
    
  if p not in like_playlists_mapped:
    if print_warn: print('Warning not in like_playlist_mapped:', p)
    pass
  
      
  # tmp reset desc
  desc=None
  if p.startswith('xr '):
    desc = f"favorites compiled from reddit sub r/{p.replace('xr ', '')}".strip()
       
  # set to public if not set and enough tracks
  privacy = None
  if set_public:
    privacy = 'PUBLIC'
    count = int(str(pl['count']).replace(',', ''))
    if count < min_count_for_public:
      print(f'Keeping private too small: {pl["count"]} < {min_count_for_public}:', p)
      privacy = None
    
    metadata = Y.playlist_get_info(pl["playlistId"])
    if metadata['privacy'] == privacy:
      print('Already public:', p)
      privacy = None

  # Apply
  if dry_run:
    print(f"Y.yt.edit_playlist(playlistId={pl['playlistId']}, privacyStatus={privacy}, description={desc})")
  else:
    status = Y.yt.edit_playlist(playlistId=pl['playlistId'],  privacyStatus=privacy, description=desc)
    print(f"Updated {p} with status={status}, privacy={privacy}, description={desc}")
    finished.append(p)


Already public: ambiant electro
Updated ambiant electro with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient
Updated ambient with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient BOC
Updated ambient BOC with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient Dream Pop Deep Sleep
Updated ambient Dream Pop Deep Sleep with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient grouper
Updated ambient grouper with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient haunting harmonious
Updated ambient haunting harmonious with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient Indie Synths
Updated ambient Indie Synths with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient japan
Updated ambient japan with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: ambient modern
U

#### (one time) check like playlists, make sure valid

In [ ]:
# dry_run = False
# log_playlist = False
# reset_description = True
# finished = []

# for i, pl in Y.playlists.iterrows():
#   p = pl['title']

#   # Filter 'valid' to like playlist
#   if p in finished:
#     print('Skip already finished:', p)
#     continue
#   if 'not like' in p:
#     # print('Skip:', p)
#     continue
#   if  '_like' in p:
#     print('Skip playlist has "_like":', p)
#     continue
#   if ' like' not in p:
#     continue
#   elif not p.endswith('like'):
#     print('Weird case not end with like:', p)
#     continue
#   if p not in like_playlists:
#     # print(p) # update _like_playlist.tsv
#     print('Skip Not logged as _like_playlist:', p)
#     continue
#   elif p not in like_playlists_mapped:
#     # print(f'{p.replace("like", "radio")}\t{p}') # update radio_to_like_pl_map
#     print('Skip Not logged as radio to like map:', p)
#     continue
#   if len(pl['playlistId']) != 34:
#     print('Skip invalid playlisdId', pl['playlistId'], p)
    
    
#   assert p.endswith(' like')
#   new_p = p.replace(' like', '')
#   if log_playlist:
#     p_dict = pl.to_dict()
#     p_dict.pop('thumbnails')
#     pprint(p_dict, sort_dicts=False)

#   # if not p.startswith('xr'): print(p)
#   desc = None
#   if reset_description:
#     # print('Reset Description for:', p, desc)
#     desc = 'favorites'
#     if p.startswith('xr '):
#       desc += f" compiled from reddit sub r/{p.replace(' xr', '')}"
    
  
#   if dry_run:
#     print(f"Y.yt.edit_playlist(playlistId={pl['playlistId']}, title={new_p}, description={desc})")
#   else:
#     status = Y.yt.edit_playlist(playlistId=pl['playlistId'], title=new_p, description=desc)
#     print(f"Updated {p} with status={status}, reset_description={reset_description}")
#     finished.append(p)
    

Skip already finished: xr PunkRock like
Skip already finished: xr 90sPunk like
Skip already finished: ambient grouper like
Skip already finished: folk country americana like
Skip already finished: rock underground like
Skip already finished: xr deepcuts like
Skip already finished: trip hop & Ambien® like
Skip already finished: xr trapmuzik like
Skip already finished: oldies fallout 40s like
Skip already finished: xr 90sRock like
Skip already finished: xr SoundsVintage like
Skip already finished: xr chillmusic like
Skip already finished: xr OutlawCountry like
Skip already finished: psych ish rock modern like
Skip already finished: rock garage like
Skip already finished: xr treemusic like
Skip already finished: brass like
Skip already finished: xr 70sMusic like
Skip already finished: xr 90sMusic like
Skip already finished: xr DreamPop like
Skip already finished: xr theOverload like
Skip already finished: xr PsychedelicRock like
Skip already finished: xr Rock like
Skip already finished: a